# Gradient Accumulation

**Interview Focus**: Simulating large batch sizes with limited GPU memory.

**Key Concepts**:
- Effective batch size = micro_batch_size x accumulation_steps x num_gpus
- Equivalence to true large-batch training
- Interaction with batch norm, learning rate scaling
- Memory vs compute tradeoff

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
import copy

## 1. The Problem

You want to train with batch size 256, but your GPU only fits batch size 32.

**Solution**: Accumulate gradients over 8 micro-batches (32 x 8 = 256), then update.

```
effective_batch_size = micro_batch_size × accumulation_steps × num_gpus
```

The gradients from each micro-batch are summed. Dividing by the number of accumulation steps gives the same result as computing gradients on the full batch.

## 2. Implementation

In [ ]:
def train_with_gradient_accumulation(model, train_loader, optimizer, criterion,
                                      accumulation_steps=4, epochs=1):
    """Training loop with gradient accumulation."""
    model.train()
    all_losses = []
    
    for epoch in range(epochs):
        optimizer.zero_grad()  # zero at the start
        running_loss = 0.0
        
        for batch_idx, (inputs, targets) in enumerate(train_loader):
            # Forward pass
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            
            # IMPORTANT: normalize loss by accumulation steps
            # This makes accumulated gradients equivalent to large-batch gradients
            loss = loss / accumulation_steps
            
            # Backward pass — gradients accumulate (not zeroed yet)
            loss.backward()
            
            running_loss += loss.item() * accumulation_steps  # track unscaled loss
            
            # Update weights every accumulation_steps batches
            if (batch_idx + 1) % accumulation_steps == 0:
                # Optional: gradient clipping on accumulated gradients
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                
                optimizer.step()
                optimizer.zero_grad()
                
                step = (batch_idx + 1) // accumulation_steps
                avg_loss = running_loss / accumulation_steps
                all_losses.append(avg_loss)
                running_loss = 0.0
        
        # Handle remaining batches (if dataset size not divisible)
        if (batch_idx + 1) % accumulation_steps != 0:
            optimizer.step()
            optimizer.zero_grad()
    
    return all_losses


print("Gradient accumulation training function defined.")
print("Key: loss = loss / accumulation_steps before backward()")

## 3. Proof of Equivalence

Show that gradient accumulation produces identical gradients to a true large batch.

In [ ]:
# Setup: simple model, fixed seed
torch.manual_seed(42)
X = torch.randn(64, 10)
y = torch.randint(0, 2, (64,))

def make_model():
    torch.manual_seed(0)
    return nn.Linear(10, 2)

criterion = nn.CrossEntropyLoss()

# Method 1: True large batch (batch_size=64)
model_large = make_model()
outputs = model_large(X)
loss = criterion(outputs, y)
loss.backward()
grad_large_batch = model_large.weight.grad.clone()

# Method 2: Gradient accumulation (4 micro-batches of 16)
model_accum = make_model()
model_accum.zero_grad()

micro_batch_size = 16
accum_steps = 4

for i in range(accum_steps):
    start = i * micro_batch_size
    end = start + micro_batch_size
    
    outputs = model_accum(X[start:end])
    loss = criterion(outputs, y[start:end])
    loss = loss / accum_steps  # normalize!
    loss.backward()

grad_accumulated = model_accum.weight.grad.clone()

# Compare
print("Large batch gradient:")
print(grad_large_batch)
print("\nAccumulated gradient:")
print(grad_accumulated)
print(f"\nMax difference: {(grad_large_batch - grad_accumulated).abs().max().item():.2e}")
print(f"Are they equal? {torch.allclose(grad_large_batch, grad_accumulated, atol=1e-6)}")

## 4. Why Not Perfectly Equivalent?

Gradient accumulation is **mathematically equivalent** for losses that are mean-reduced (like CrossEntropyLoss with reduction='mean'). However:

1. **Batch Normalization**: BN statistics are computed per micro-batch, not per effective batch. This means the running mean/variance will be slightly different. Fix: use LayerNorm or GroupNorm instead.

2. **Dropout**: Different random masks per micro-batch vs one mask for the large batch. Minor effect.

3. **Numerical precision**: Accumulating FP16 gradients over many steps can cause rounding errors.

In [ ]:
# Demonstrate BatchNorm difference

class SmallBNNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(10, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Linear(32, 2)
        )
    def forward(self, x):
        return self.layers(x)

torch.manual_seed(42)
X = torch.randn(64, 10)
y = torch.randint(0, 2, (64,))

# Large batch
torch.manual_seed(0)
model_bn_large = SmallBNNet()
model_bn_large.train()
out_large = model_bn_large(X)
bn_mean_large = model_bn_large.layers[1].running_mean.clone()

# Accumulated micro-batches
torch.manual_seed(0)
model_bn_accum = SmallBNNet()
model_bn_accum.train()
for i in range(4):
    _ = model_bn_accum(X[i*16:(i+1)*16])
bn_mean_accum = model_bn_accum.layers[1].running_mean.clone()

print("BN running_mean with large batch (64):")
print(f"  {bn_mean_large[:5].tolist()}")
print("BN running_mean with 4 micro-batches (16 each):")
print(f"  {bn_mean_accum[:5].tolist()}")
print(f"\nMax difference: {(bn_mean_large - bn_mean_accum).abs().max().item():.6f}")
print("\nBN statistics differ because they are computed per micro-batch.")
print("Recommendation: use LayerNorm for gradient accumulation (standard in transformers).")

## 5. Training Comparison: Large Batch vs Accumulated

In [ ]:
# Create dataset
torch.manual_seed(42)
n_samples = 1000
X_all = torch.randn(n_samples, 20)
w_true = torch.randn(20, 1)
y_all = (X_all @ w_true + 0.1 * torch.randn(n_samples, 1)).squeeze()
y_cls = (y_all > y_all.median()).long()

dataset = TensorDataset(X_all, y_cls)

# Model factory
def make_classifier():
    torch.manual_seed(0)
    return nn.Sequential(
        nn.Linear(20, 64),
        nn.LayerNorm(64),  # LayerNorm instead of BatchNorm
        nn.ReLU(),
        nn.Linear(64, 2)
    )

criterion = nn.CrossEntropyLoss()

# --- Train 1: True large batch (64) ---
model_large = make_classifier()
optimizer_large = torch.optim.SGD(model_large.parameters(), lr=0.01)
loader_large = DataLoader(dataset, batch_size=64, shuffle=True,
                          generator=torch.Generator().manual_seed(42))

losses_large = []
for epoch in range(20):
    epoch_loss = 0
    n_batches = 0
    for inputs, targets in loader_large:
        outputs = model_large(inputs)
        loss = criterion(outputs, targets)
        optimizer_large.zero_grad()
        loss.backward()
        optimizer_large.step()
        epoch_loss += loss.item()
        n_batches += 1
    losses_large.append(epoch_loss / n_batches)

# --- Train 2: Gradient accumulation (batch=16, accum=4 → effective=64) ---
model_accum = make_classifier()
optimizer_accum = torch.optim.SGD(model_accum.parameters(), lr=0.01)
loader_small = DataLoader(dataset, batch_size=16, shuffle=True,
                          generator=torch.Generator().manual_seed(42))

accum_steps = 4
losses_accum = []

for epoch in range(20):
    epoch_loss = 0
    n_updates = 0
    optimizer_accum.zero_grad()
    
    for batch_idx, (inputs, targets) in enumerate(loader_small):
        outputs = model_accum(inputs)
        loss = criterion(outputs, targets) / accum_steps
        loss.backward()
        epoch_loss += loss.item() * accum_steps
        
        if (batch_idx + 1) % accum_steps == 0:
            optimizer_accum.step()
            optimizer_accum.zero_grad()
            n_updates += 1
    
    # Handle remaining
    if (batch_idx + 1) % accum_steps != 0:
        optimizer_accum.step()
        optimizer_accum.zero_grad()
        n_updates += 1
    
    losses_accum.append(epoch_loss / (batch_idx + 1))

# --- Train 3: Small batch without accumulation (batch=16) ---
model_small = make_classifier()
optimizer_small = torch.optim.SGD(model_small.parameters(), lr=0.01)
loader_small2 = DataLoader(dataset, batch_size=16, shuffle=True,
                           generator=torch.Generator().manual_seed(42))

losses_small = []
for epoch in range(20):
    epoch_loss = 0
    n_batches = 0
    for inputs, targets in loader_small2:
        outputs = model_small(inputs)
        loss = criterion(outputs, targets)
        optimizer_small.zero_grad()
        loss.backward()
        optimizer_small.step()
        epoch_loss += loss.item()
        n_batches += 1
    losses_small.append(epoch_loss / n_batches)

print(f"Final loss — Large batch (64):       {losses_large[-1]:.4f}")
print(f"Final loss — Accum (16x4=64):        {losses_accum[-1]:.4f}")
print(f"Final loss — Small batch (16):        {losses_small[-1]:.4f}")

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(losses_large, 'b-', linewidth=2, label='Large batch (64)', marker='o', markersize=4)
plt.plot(losses_accum, 'r--', linewidth=2, label='Accum (16 x 4 = 64)', marker='s', markersize=4)
plt.plot(losses_small, 'g:', linewidth=2, label='Small batch (16)', marker='^', markersize=4)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Gradient Accumulation: Equivalent to Large Batch')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Large batch and accumulated curves should track closely.")
print("Small batch (without accumulation) has more updates per epoch → different dynamics.")

## 6. Memory Analysis

In [ ]:
def estimate_training_memory(model, batch_size, seq_len=512, dtype_bytes=4):
    """Rough memory estimate for training.
    
    Components:
    - Parameters
    - Gradients (same size as params)
    - Optimizer states (2x params for Adam)
    - Activations (proportional to batch_size)
    """
    param_bytes = sum(p.numel() * dtype_bytes for p in model.parameters())
    grad_bytes = param_bytes  # same size
    optim_bytes = param_bytes * 2  # Adam: mean + variance
    
    # Rough activation estimate: proportional to batch_size
    # Each layer stores input for backward pass
    activation_bytes = 0
    for name, module in model.named_modules():
        if isinstance(module, nn.Linear):
            activation_bytes += batch_size * module.in_features * dtype_bytes
    
    total = param_bytes + grad_bytes + optim_bytes + activation_bytes
    
    return {
        'params_mb': param_bytes / 1e6,
        'grads_mb': grad_bytes / 1e6,
        'optim_mb': optim_bytes / 1e6,
        'activations_mb': activation_bytes / 1e6,
        'total_mb': total / 1e6,
    }


model = nn.Sequential(
    nn.Linear(784, 1024),
    nn.ReLU(),
    nn.Linear(1024, 1024),
    nn.ReLU(),
    nn.Linear(1024, 10),
)

print(f"{'Batch Size':>12} {'Params':>10} {'Grads':>10} {'Optim':>10} {'Activations':>12} {'Total':>10}")
print("-" * 68)
for bs in [16, 32, 64, 128, 256]:
    mem = estimate_training_memory(model, bs)
    print(f"{bs:>12} {mem['params_mb']:>8.1f}MB {mem['grads_mb']:>8.1f}MB "
          f"{mem['optim_mb']:>8.1f}MB {mem['activations_mb']:>10.1f}MB {mem['total_mb']:>8.1f}MB")

print("\nKey insight: Params/grads/optim are FIXED. Only activations scale with batch size.")
print("Gradient accumulation uses the memory of the micro-batch, not the effective batch.")

## 7. Complete Training Template with Gradient Accumulation + AMP

In [ ]:
from torch.amp import autocast, GradScaler

def train_full_template(model, train_loader, optimizer, scheduler, criterion,
                         accumulation_steps=4, max_grad_norm=1.0,
                         device='cpu', dtype=torch.bfloat16, epochs=5):
    """Production training loop: gradient accumulation + AMP + grad clipping."""
    
    model.to(device)
    use_scaler = (dtype == torch.float16 and device != 'cpu')
    scaler = GradScaler(device, enabled=use_scaler)
    
    global_step = 0
    
    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad(set_to_none=True)
        epoch_loss = 0
        
        for batch_idx, (inputs, targets) in enumerate(train_loader):
            inputs = inputs.to(device)
            targets = targets.to(device)
            
            with autocast(device_type=device, dtype=dtype):
                outputs = model(inputs)
                loss = criterion(outputs, targets) / accumulation_steps
            
            scaler.scale(loss).backward()
            epoch_loss += loss.item() * accumulation_steps
            
            if (batch_idx + 1) % accumulation_steps == 0:
                # Unscale before clipping
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
                
                scaler.step(optimizer)
                scaler.update()
                
                if scheduler is not None:
                    scheduler.step()
                
                optimizer.zero_grad(set_to_none=True)
                global_step += 1
        
        avg_loss = epoch_loss / (batch_idx + 1)
        print(f"Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.4f}, Steps: {global_step}")


# Demo
torch.manual_seed(42)
X = torch.randn(500, 20)
y = torch.randint(0, 2, (500,))
loader = DataLoader(TensorDataset(X, y), batch_size=16, shuffle=True)

model = nn.Sequential(nn.Linear(20, 64), nn.ReLU(), nn.Linear(64, 2))
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

train_full_template(
    model, loader, optimizer, scheduler=None, criterion=criterion,
    accumulation_steps=4, device='cpu', dtype=torch.bfloat16, epochs=5
)

## 8. Learning Rate Scaling Rule

When you increase effective batch size, you should often scale the learning rate.

**Linear scaling rule** (Goyal et al., 2017):
```
lr_new = lr_base * (new_batch_size / base_batch_size)
```

Example: If lr=0.1 works for batch_size=256, use lr=0.4 for batch_size=1024.

**Warmup is critical** when using large learning rates — prevents early divergence.

In [ ]:
base_lr = 1e-3
base_batch_size = 32

print(f"{'Effective BS':>15} {'LR (linear scale)':>20} {'Accum Steps (micro=32)':>25}")
print("-" * 65)
for effective_bs in [32, 64, 128, 256, 512, 1024]:
    scaled_lr = base_lr * (effective_bs / base_batch_size)
    accum_steps = effective_bs // base_batch_size
    print(f"{effective_bs:>15} {scaled_lr:>20.5f} {accum_steps:>25}")

## Interview Questions & Answers

---

**Q: Why gradient accumulation?**

To simulate a larger effective batch size when GPU memory is limited. You get the same gradient signal as a large batch, using only the memory of a micro-batch. The tradeoff is more compute time (same data processed sequentially instead of in parallel).

---

**Q: Does it perfectly match large batch training?**

**Mathematically yes** for mean-reduced losses with no stochastic operations between micro-batches. **In practice, almost yes**, with these caveats:
- BatchNorm computes stats per micro-batch (use LayerNorm instead)
- Dropout generates different masks per micro-batch (minor effect)
- FP16 accumulation can have rounding errors (use FP32 gradient accumulation)

---

**Q: Memory vs compute tradeoff?**

- **Memory**: Only the micro-batch's activations are in memory at once. Huge savings.
- **Compute**: Same total FLOPs, but:
  - No parallelism across micro-batches (sequential on same GPU)
  - Slightly more overhead (multiple forward/backward passes)
  - In DDP: fewer AllReduce calls (only after accumulation_steps), which can speed things up

---

**Q: Interaction with batch norm?**

BatchNorm computes mean/variance per micro-batch, not per effective batch. Solutions:
1. Use LayerNorm instead (standard in transformers)
2. Use SyncBatchNorm across GPUs (for vision models)
3. Use GroupNorm (independent of batch size)

---

**Q: The crucial detail people forget?**

`loss = loss / accumulation_steps` before `.backward()`. Without this, accumulated gradients are `accumulation_steps` times too large.

## Quick Reference

```python
accumulation_steps = 8
optimizer.zero_grad()

for batch_idx, (x, y) in enumerate(loader):
    loss = model(x, y) / accumulation_steps  # normalize!
    loss.backward()
    
    if (batch_idx + 1) % accumulation_steps == 0:
        clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()
```

**Effective batch size**: `micro_batch_size * accumulation_steps * num_gpus`

**Learning rate**: Scale linearly with effective batch size, use warmup.